In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = "../.."
FIG_DIR = "../figures"
import os
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
RUN = f"{REPO}/results/20260813_053524_36857/csv"
pipeline = pd.read_csv(f"{RUN}/pipeline_summary.csv")
ptq = pd.read_csv(f"{RUN}/ptq_summary.csv")
qat = pd.read_csv(f"{RUN}/qat_summary.csv")
pipeline

,model,dataset,best_train_acc,best_val_acc,ptq_val_acc,qat_val_acc,fp32_fps,ptq_fps,qat_fps,wall_time_min,status
0,cnn,CIFAR10,82.46,81.30,71.08,84.46,1027.6,397.1,390.4,5.2,ok
1,resnet18_no_weights,CIFAR10,89.71,84.18,82.76,90.51,306.2,79.5,37.7,11.5,ok
2,resnet50_no_weights,CIFAR10,84.56,81.14,79.54,86.80,133.6,27.6,13.8,24.2,ok
3,cnn,IMAGENET100,40.42,43.04,25.06,51.18,861.6,322.0,314.4,112.2,ok
4,resnet18_no_weights,IMAGENET100,84.65,80.06,71.30,82.52,292.9,74.9,74.7,343.9,ok
5,resnet50_no_weights,IMAGENET100,83.08,80.02,64.06,83.22,127.8,26.5,13.7,537.9,ok


In [ ]:
merged = pipeline.merge(ptq[["model", "dataset", "acc_drop"]], on=["model", "dataset"])
merged = merged.merge(qat[["model", "dataset", "acc_recovered"]], on=["model", "dataset"])

MODEL_LABEL = {"cnn": "CNN", "resnet18_no_weights": "ResNet-18", "resnet50_no_weights": "ResNet-50"}
DATASET_LABEL = {"CIFAR10": "CIFAR10", "IMAGENET100": "ImageNet100"}
merged["Modell"] = merged["model"].map(MODEL_LABEL)
merged["Datensatz"] = merged["dataset"].map(DATASET_LABEL)

table1 = merged[["Datensatz", "Modell", "best_val_acc", "ptq_val_acc", "qat_val_acc", "acc_drop", "acc_recovered"]]
table1.columns = ["Datensatz", "Modell", "FP32", "PTQ", "QAT", "PTQ-Verlust", "QAT-Erholung"]
table1 = table1.sort_values(["Datensatz", "Modell"], ascending=[False, True]).reset_index(drop=True)
table1

,Datensatz,Modell,FP32,PTQ,QAT,PTQ-Verlust,QAT-Erholung
0,ImageNet100,CNN,43.04,25.06,51.18,17.98,26.12
1,ImageNet100,ResNet-18,80.06,71.30,82.52,8.76,11.22
2,ImageNet100,ResNet-50,80.02,64.06,83.22,15.96,19.16
3,CIFAR10,CNN,81.30,71.08,84.46,10.22,13.38
4,CIFAR10,ResNet-18,84.18,82.76,90.51,1.42,7.75
5,CIFAR10,ResNet-50,81.14,79.54,86.80,1.60,7.26


In [ ]:
table1_rounded = table1.copy()
for c in ["FP32", "PTQ", "QAT", "PTQ-Verlust", "QAT-Erholung"]:
    table1_rounded[c] = table1_rounded[c].round(2)
table1_rounded.to_csv(f"{FIG_DIR}/tab_01_accuracy_overview.csv", index=False)
with open(f"{FIG_DIR}/tab_01_accuracy_overview.tex", "w") as f:
    f.write(table1_rounded.to_latex(index=False, float_format="%.2f"))
table1_rounded

,Datensatz,Modell,FP32,PTQ,QAT,PTQ-Verlust,QAT-Erholung
0,ImageNet100,CNN,43.04,25.06,51.18,17.98,26.12
1,ImageNet100,ResNet-18,80.06,71.30,82.52,8.76,11.22
2,ImageNet100,ResNet-50,80.02,64.06,83.22,15.96,19.16
3,CIFAR10,CNN,81.30,71.08,84.46,10.22,13.38
4,CIFAR10,ResNet-18,84.18,82.76,90.51,1.42,7.75
5,CIFAR10,ResNet-50,81.14,79.54,86.80,1.60,7.26
